In [ ]:
%pip install -r requirements.txt

In [23]:
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from azure.ai.projects.models import (
    EvaluationRule,
    ContinuousEvaluationRuleAction,    
    EvaluationRuleFilter,
    EvaluationRuleEventType,
    AgentVersionDetails,
    AzureAIDataSourceConfig,
    EvaluationTaxonomy,
    AzureAIAgentTarget,
    AgentTaxonomyInput,
    TestingCriterionAzureAIEvaluator,
    RedTeamEvalRunDataSource,
    Schedule,
    RecurrenceTrigger,
    DailyRecurrenceSchedule,
    EvaluationScheduleTask,
    RiskCategory,    
)
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam, SourceFileID
)
from openai.types.eval_create_params import DataSourceConfigCustom
from azure.ai.projects.models import (
    Schedule, RecurrenceTrigger, DailyRecurrenceSchedule, EvaluationScheduleTask
)
import os
import time

In [2]:
load_dotenv(override=True)

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_deployment = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

In [6]:
project = AIProjectClient(
    endpoint=endpoint,
    credential=AzureCliCredential()
)

client = project.get_openai_client()

In [3]:
data_source_config = {"type": "azure_ai_source", "scenario": "responses"}

In [8]:
testing_criteria = [
    # RAG evaluators
    {"type": "azure_ai_evaluator", "name": "groundedness", "evaluator_name": "builtin.groundedness",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"}},
    {"type": "azure_ai_evaluator", "name": "relevance", "evaluator_name": "builtin.relevance",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"}},
    # General purpose evaluators
    {"type": "azure_ai_evaluator", "name": "fluency", "evaluator_name": "builtin.fluency",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"response": "{{item.response}}"}},
    {"type": "azure_ai_evaluator", "name": "coherence", "evaluator_name": "builtin.coherence",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"}},
    # Agent system evaluators
    {"type": "azure_ai_evaluator", "name": "task_completion", "evaluator_name": "builtin.task_completion",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"}},
    {"type": "azure_ai_evaluator", "name": "task_adherence", "evaluator_name": "builtin.task_adherence",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"}},
    {"type": "azure_ai_evaluator", "name": "intent_resolution", "evaluator_name": "builtin.intent_resolution",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"}},
    # Agent process evaluators (tool-related)
    {"type": "azure_ai_evaluator", "name": "tool_selection", "evaluator_name": "builtin.tool_selection",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}", "tool_definitions": "{{item.tool_definitions}}"}},
    {"type": "azure_ai_evaluator", "name": "tool_input_accuracy", "evaluator_name": "builtin.tool_input_accuracy",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}", "tool_definitions": "{{item.tool_definitions}}"}},
    {"type": "azure_ai_evaluator", "name": "tool_output_utilization", "evaluator_name": "builtin.tool_output_utilization",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}", "tool_definitions": "{{item.tool_definitions}}"}},
    {"type": "azure_ai_evaluator", "name": "tool_call_success", "evaluator_name": "builtin.tool_call_success",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"response": "{{item.response}}"}},
    {"type": "azure_ai_evaluator", "name": "tool_call_accuracy", "evaluator_name": "builtin.tool_call_accuracy",
     "initialization_parameters": {"deployment_name": model_deployment},
     "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}", "tool_definitions": "{{item.tool_definitions}}"}},
]

In [9]:
eval_object = client.evals.create(
    name="Continuous Evaluation",
    data_source_config=data_source_config,  # type: ignore
    testing_criteria=testing_criteria,  # type: ignore
)

In [ ]:
print(f"Evaluation created (id: {eval_object.id}, name: {eval_object.name})")

In [38]:
rules = project.evaluation_rules.list()

for rule in rules:
    print(rule.__dict__)    

{'_data': {'id': 'continuous_evaluation_wealthagent_users', 'displayName': 'Continuous evaluation for agent WealthAgent', 'description': 'An eval rule that runs for agent WealthAgent response completions', 'systemData': {'createdAt': '06/19/2026 13:33:15', 'createdBy': '', 'modifiedAt': '', 'modifiedBy': ''}, 'action': {'type': 'continuousEvaluation', 'evalId': 'eval_1dd2c478044e41b6b79ef6d1edb4479d', 'maxHourlyRuns': 10, 'samplingRate': None}, 'filter': {'agentName': 'WealthAgent'}, 'eventType': 'responseCompleted', 'enabled': True}}


In [39]:
project.evaluation_rules.delete('continuous_evaluation_wealthagent_users')

In [ ]:
continuous_eval_rule = project.evaluation_rules.create_or_update(
    id="my-continuous-eval-rule",
    evaluation_rule=EvaluationRule(
        display_name="My Continuous Eval Rule",
        description="An eval rule that runs on agent response completions",
        action=ContinuousEvaluationRuleAction(eval_id=eval_object.id, max_hourly_runs=10,sampling_rate=1), # value between 0.0 and 1.0
        event_type=EvaluationRuleEventType.RESPONSE_COMPLETED,
        filter=EvaluationRuleFilter(agent_name='WealthAgent'),
        enabled=True,
    ),
)
print(
    f"Continuous Evaluation Rule created (id: {continuous_eval_rule.id}, name: {continuous_eval_rule.display_name})"
)

Continuous Evaluation Rule created (id: my-continuous-eval-rule, name: My Continuous Eval Rule)


In [17]:
eval_run_list = client.evals.runs.list(
    eval_id=eval_object.id,
    order="desc",
    limit=10,
)

if len(eval_run_list.data) > 0 and eval_run_list.data[0].report_url:
    print(f"Report URL: {eval_run_list.data[0].report_url}")

In [ ]:
ds = project.datasets.list()

for d in ds:
    print(d.name)

In [21]:
dataset = project.datasets.upload_file(
    name="cicd-eval-data",
    version="1",
    file_path="dataset/cicd.json",
)

In [ ]:
print(f"Dataset uploaded: {dataset.id}")

In [24]:
data_source_config_custom = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "tool_definitions": {"type": "string"},
        },
        "required": [],
    },
    include_sample_schema=True,
)

In [25]:
eval_object = client.evals.create(
    name="Scheduled Evaluation",
    data_source_config=data_source_config_custom,  # type: ignore
    testing_criteria=testing_criteria,  # type: ignore  <-- reusing your existing criteria
)
print(f"Eval created: {eval_object.id}")

Eval created: eval_749df0f93d844d15b8eeb7c51986b233


In [29]:
# 3. Create the eval run definition referencing the uploaded dataset
eval_run_object = {
    "eval_id": eval_object.id,
    "name": "cicd-dataset-run",
    "data_source": {
        "type": "responses",
        "source": {"type": "file_id", "id": dataset.id},
        "model": "WealthAgent",
        "input_messages": {
            "type": "template",
            "template": [{"role": "user", "content": "{{item.query}}"}],
        },
    },
}